### Machine Learning: In-Season Projection and Cluster-Based Role Analysis
- Since there is only one years worth of data, time series analysis cannot be used since it requires years and years worth of data
- Instead we will use unsupervised learning models, specifically K-Means Clustering to group players into roles based on the new statistics calculated
- Then using regression to project where players goal tally based on where they should end up by the end of the season based on their scoring efficiency (TFE), 
    - to see if players are underperforming or overperforming, and by how much

In [4]:
import pandas as pd

K-Means Clustering

In [3]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

In [5]:
comparison_df = pd.read_csv('data/adv_stats_inc/master_player_comparison.csv')

features = ['Possession_USG_pct', 'Attacking_USG_pct', 'Total_Offensive_Goals_Added', 'TFE']

# Clean data (Removing NaNs from TFE and filter for significant minutes)
ml_df = comparison_df[(comparison_df['90s'] >= 5) & (comparison_df['TFE'].notna())].copy()

# Standardize the data (Crucial for K-Means)
scaler = StandardScaler()
scaled_features = scaler.fit_transform(ml_df[features])

# Apply K-Means (Setting 5 clusters as a baseline for player roles)
kmeans = KMeans(n_clusters=5, random_state=246)
ml_df['Player_Role_Cluster'] = kmeans.fit_transform(scaled_features).argmax(axis=1)

print("Players clustered into 5 distinct usage roles.")

Players clustered into 5 distinct usage roles.


Similarity Search
- to find the closest statistical matches for any player

In [ ]:
from scipy.spatial.distance import cdist

def find_statistical_twins(player_name, df, num_twins=5):
    """
    Finds the most similar players based on the Euclidean distance 
    of their scaled Usage and Efficiency features.
    """
    # 1. Get the target player's features
    target_player = df[df['Player'] == player_name]
    if target_player.empty:
        return f"Player '{player_name}' not found in the ML dataset."
    
    # 2. Extract features for comparison (must match the ones used in clustering)
    features = ['Possession_USG_pct', 'Attacking_USG_pct', 'Total_Offensive_Goals_Added', 'TFE']
    
    # 3. Calculate distance between this player and all others
    # We use the scaled versions of these features for accuracy
    source_features = scaler.transform(df[features])
    target_features = scaler.transform(target_player[features])
    
    distances = cdist(target_features, source_features, metric='euclidean')[0]
    
    # 4. Add distances to a temporary copy of the dataframe
    results = df.copy()
    results['Similarity_Score'] = distances
    
    # 5. Sort by distance (lowest is most similar) and exclude the player themselves
    twins = results[results['Player'] != player_name].nsmallest(num_twins, 'Similarity_Score')
    
    return twins[['Player', 'Squad', 'Pos', 'Similarity_Score'] + features]

# Example: Find players similar to Vinicius Júnior
twins_df = find_statistical_twins("Vinicius Júnior", ml_df)
print(twins_df)

             Player           Squad    Pos  Similarity_Score  \
206    Dilane Bakwa      Strasbourg  MF,DF          0.881067   
1606  Omar Marmoush  Eint Frankfurt     FW          0.900165   
2264    Bukayo Saka         Arsenal  FW,MF          0.907645   
27    Karim Adeyemi        Dortmund  FW,MF          0.912272   
1440    Rafael Leão           Milan     FW          0.975841   

      Possession_USG_pct  Attacking_USG_pct  Total_Offensive_Goals_Added  \
206             8.182411          23.334078                     6.211746   
1606            6.642655          26.331253                     6.262543   
2264            8.007049          22.167093                     5.854621   
27              5.586550          20.632128                     5.085858   
1440            7.276697          19.690269                     4.402088   

           TFE  
206   1.276596  
1606  2.054795  
2264  1.000000  
27    1.272727  
1440  1.052632  


In-Season Goal Projections
- build a simple linear model to project total goals
- project their current rate across a full 38-game season (standard for most top 5 leagues)

In [7]:
from sklearn.linear_model import LinearRegression

def project_season_totals(df):
    """
    Projects end-of-season goals based on current npxG and TFE.
    Assumes a standard 38-game season.
    """
    # Calculate Goals per 90
    df['Gls_per_90'] = (df['Attacking_USG_pct'] * df['TFE']) / 100 # Simplified proxy
    
    # Project to 38 games (3420 minutes)
    df['Projected_Final_Goals'] = df['Gls_per_90'] * 38
    
    # Calculate 'Performance Gap' 
    # (Are they overperforming or underperforming their projectable metrics?)
    df['Goal_Variance_Projection'] = df['Projected_Final_Goals'] - (df['Total_Offensive_Goals_Added'])
    
    return df

ml_projection_df = project_season_totals(ml_df)
print(ml_projection_df[['Player', 'Projected_Final_Goals', 'Goal_Variance_Projection']].sort_values(by='Projected_Final_Goals', ascending=False).head(10))

               Player  Projected_Final_Goals  Goal_Variance_Projection
1369      Ismaël Koné              59.832294                 59.328053
2388      Moses Simon              38.530856                 40.829606
741       Jeremy Doku              36.831127                 37.508334
2291     Iván Sánchez              33.224758                 34.810017
804     Chidera Ejuke              30.817950                 32.627937
722      Yan Diomandé              29.890930                 29.618499
1423    Tariq Lamptey              29.451129                 29.143175
1135          Hernani              29.380347                 30.195281
816        Elif Elmas              28.869845                 31.628323
1653  Stephy Mavididi              28.315005                 31.961619
